# FLIDS — Experiment 08 + 09 | Complete Kaggle AutoRun

**Only dataset input required:** attach the original NF-UQ-NIDS-v2 combined CSV or all four matching NF-v2 source CSVs. A matching prepared Experiment 02 cache/fingerprints pair is optional. No previous notebook or Experiment 08 result archive is required.

1. Prepare the original 96,000-row reservoir once (12,000 per domain/class).
2. Reproduce Experiment 08: four held-out domains × five seeds × four algorithms × four candidates = **320 candidate trainings**.
3. Retune FedProx: four domains × five seeds × four fresh candidates = **80 candidate trainings**. Client LR {0.001, 0.003}, μ {0.001, 0.01}.
4. Audit original and retuned scores at source-calibrated FPR 0.1%, 1%, 5%. Compare old/new FedProx and generate final four-method tables.

**400 candidate trainings, 30 rounds each (12,000 round records), 80 final selected models, 240 planned final operating points.** Source-only tuning (pAUC@5%) and separate source calibration, with no target data used for selection. All four original algorithms and the historical FedProx grid are preserved so before/after results can be compared.

Kaggle: add the dataset, select GPU accelerator, and Run All. Internet not required if dependencies are installed. A single GPU is used sequentially. The initial full CSV scan may be substantial. `SMOKE_TEST=False` is mandatory for research results.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil, subprocess, time, base64, zlib
from IPython.display import FileLink, display
SMOKE_TEST = False
NF_UQ_DATA = ''  # Optional exact CSV path or folder (otherwise scan Kaggle inputs)
CACHE_DIR = ''   # Optional folder containing both original Experiment 02 cache files
RESUME_DIR = ''  # Optional path to a previous full flids_exp08_09_work directory
AUTO_RESUME = True
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
WORK = BASE / ('flids_exp08_09_smoke' if SMOKE_TEST else 'flids_exp08_09_work')
WORK.mkdir(parents=True,exist_ok=True)
INPUT = Path('/kaggle/input')
import torch
print('Work:',WORK,'| smoke:',SMOKE_TEST,'| CUDA:',torch.cuda.is_available())
if not SMOKE_TEST and not torch.cuda.is_available(): print('WARNING: CPU execution may be slow. Enable GPU accelerator in Kaggle.')

In [ ]:
# Resolve optional resume folder before writing engine files.
def unique_path(paths,label):
    paths=list(dict.fromkeys(p.resolve() for p in paths))
    if len(paths)>1: raise RuntimeError(f'Multiple {label}: {paths}. Set explicit path in configuration.')
    return paths[0] if paths else None
if not SMOKE_TEST:
    if AUTO_RESUME and not (WORK/'stage08_identity.json').exists():
        prev=Path(RESUME_DIR) if RESUME_DIR else unique_path((p.parent for p in INPUT.rglob('stage08_identity.json') if p.parent.name=='flids_exp08_09_work'),'resume work directories')
        if prev is not None:
            if not (prev/'stage08_identity.json').is_file(): raise RuntimeError('RESUME_DIR must be the complete previous work folder')
            if any(WORK.iterdir()): raise RuntimeError('Local work folder is not empty; cannot safely merge resume artifacts')
            shutil.copytree(prev,WORK,dirs_exist_ok=True)
            print('RESTORED_WORK_FOLDER',prev)
    pair=['flids_exp02_cache.npz','flids_exp02_fingerprints.npy']
    present=[(WORK/n).exists() for n in pair]
    if any(present) and not all(present): raise RuntimeError('Incomplete local cache pair; restore both files')
    if not all(present):
        if CACHE_DIR:
            source=Path(CACHE_DIR)
            if not all((source/n).is_file() for n in pair): raise FileNotFoundError('CACHE_DIR must contain both cache files')
        else:
            source=unique_path((p.parent for p in INPUT.rglob(pair[0]) if (p.parent/pair[1]).is_file()),'prepared cache pairs')
        if source:
            for n in pair: shutil.copy2(source/n,WORK/n)
            print('RESTORED_CACHE_PAIR',source)
    if NF_UQ_DATA:
        if not Path(NF_UQ_DATA).exists(): raise FileNotFoundError(NF_UQ_DATA)
        os.environ['NF_UQ_DATA']=NF_UQ_DATA

In [ ]:
PAYLOADS = {'engine02.py': 'eNqtfHtz40aS5//8FBhv7ACQSmiS/XAvaTiOLVFtnSVKFiV7vAwGAiJAChYIYACQEqXTffbLRxVQIKme3tudGLeAqkI9svLxy6ws/vDDD8OnLMyjZZiURrvbM4oyj2ZlvDGKdJXPwqP7MA6O0lVpjMLyNE4fjd+7xun50dnJ2LgLk9n90s8fnNYwWcRRcX+UJvGmbySpUfr5IiyPgnTpR4lRrpIoWRhpLsuN8MlfZnFYGFDJAxllDi2hldM6jdahsQzL+zQohDGHamgWhFkI/8AsZfvAL/0CurpP4wDmhy3xuyIM8SuY6dHp1bURrv145ZdRmjitH374oRUtszQvjbRozWE2Dzi+ZV5eXHmj2wvv5pfr4eBkbArz8mo4+nI+GG8VX/x6vlUCb8N/XF03Su1eWjhhso7yNJk8TF2zY6pxYfWZnxehmBVrce8X93F0J/4q0kSUsAUCaDAL7/zZg3iOsnkUh2Lur+Ly3k+COMxFDn/TZWuep0sj80v82JAdX8ErV8zSOA5nuOJCVR6nq6SE74OQugtgg9WEktUy2xh+YSSZKspgFCiA/2eBKivTfCb7p0fVc5JohU6SqPL5KqEp+DH2c8qNioc49PPEWaZBGHtFKKepvvmap6tsfL+az+NwnMVRufVViJxZrSlPZ56/mnnFLM1D4a/D3F+EXpaHs6iATrm8pZMPtsS/i0PLj2OvvM9DPyjcm3wV2q3ry8sbFylowcYBe8LeWebpOfC4B5vb7nrYADb73YO/WMThu8c0fwBONY1oToS3dmpsJ3yKirKwbCOMi9Aw3y2T8h2y7LvwKYMe43Tmx6Ztty5vb1zs/p0asBJHHDcsYP6F2YdWzvIhiHIL2AfqeOaCBvHSB7mO48HxL0PZ2zyOgsLjwWb+7D50kuzZbJ1cXgzORmN3Yo5Oj25H4z+ORl86H5GRT49u0tHRWXrDL1/Sm/rleDw8Oj47RqnvtjufzWnrdDi4ub0eYkfnH7zx9bF3dXmNreHtZHyj3q6uL28ujy/PseJHj97g8WzkffnzZkiCdntTPUPx1a83qlQ+np5f/uGd3F4Pbs4uR97F2fn52Xh4fDki6bs5vvJOzwdf8fn4/Gw4uvH0ovHw+vfhdaOo6ulspL/BgCjeMIObG5zsxeAf8un8cvR1CAuiecCcsNtfYHFbZfjp2RW+eOfDkeyhUYA0urkk4vACqnVjEdRgg62a6+HN9WA0vji7uRmeeBrZdioktZrlOnF3a+Qn2rwGv39FJXZ5+/WXKyKINrOdOtR42IV3e4VtOt3P1VhVFRZCXffjp906LIS6j53ubh0WYp/t7oc9nWIp1n7s1LW4x38AHYjso60C3t2zY1DzN39eDdXz2dXvH1TByWjs/XY7vP7TOztpvGr1wBAekPCP4TXy5c2Vd3x5cTGA7QLawvPJEOTiYnjzy+UJisVVmsZhcHF+hWJAz+fR4r78+uUCuQrFn+tOw2CwXlTPV3n6hC/T1vHpVxdVtUUmzZ20xYeueP+h/aPodN+Lbrv7aSrYvhauFGshraYrpyHmoV+u8rBwlbwKtp7eLI5IjbwXoHUT+OJ9W5BO8sIsnd0Xbkfc+eXs3iui59CFvRL8hUfNQaNkLmySiHPXabc7IoNZe8sVvHQEm3hvnuUwaap16L+PU7EEC79cLT1ADtEi8cAEe1Dvdtrttlj6T9D5Y+GB/vN4XTBLv4CpdLFeThzseRSQQffmYC3xwXW6MHkk7uJu6ZV5COvtwHp8sEkACdxvaFYH7KwpClCPS981b3/TUE7fCFJAMoA3VndFGZWrMjSGwSI8OgO1CMDDFHlUPHjQAoabmDdgUQCURPN5mO+iFDCneQi4yZB0F9QzTH8FFjLNF34SPfvSYALSidMNzrNA8QRMUEbzKAwM2McwX6dRbszApC0APYG5A3oADAsNsHHQAE0jtKzM4LvB7dX1MUrPNiSrEBjAoNi/AytlrAr4FNGaRGLCKIAh6IFMtlGZbEFQDlZcIPqiaTJaRPxn1FuktVHUXKx8gDEl0Kq8B7TmL0OFCQGuaZIBANAHBIP/GmTLjTApwiUYcCOdU7+I9cAa+mUIY9KcYfpydTRfnNfpKo4R5MDOzwwpDQAUgXp5lkeJ3Bgf4MTjUQogIvYzA5l5wztUhAWREfidvqxx6CxEmwOblMNqn2HkO9hnIBZQNA//uYpy3nsjA9GH4fpQCqsIoCFwwd2mRMovI0BQYYmgwcDB7wGTmFO7BbDiDSAyvrj8dWjaLmLKHigIZ5XBKGGlInY0wqTXnSoR7+7KMCjobwhed49cff6WEH9z6sMhYWOY9cSk+ZpTdwJbYD3bBoLxZ+Q9DTnv+XrqFAgMLSC9PW21AM8CB61DDxG0hYhYpHd/2T3CZPhqO495VIZeGT6VFjZyAsC7hQWtRETeBBBFomIXPB9bwMamAeyXa67K+dFn05aDzEMPdyfMud9ey3iMynsjBWbggc3c3P74qIgWUJjnKWhCE4Qz9pFrkvAR+cQ1TRvB8byXh8BbCXACTBL8AifngeY2D56k+ZJ5zCpt1dg0nb/SKLFmRLoZ+VFlDg0c0F7wsY3YdOZEhR8D91uyq5mfpEmEWp73GTtsGYXbGKKFn5qrpHg0qVs1pOKo9pRblGnCDUj3mFFa7m/fke3vVIN/0b4r28+KULaHFZqzaAaAltDn3q/ew1eyaJQmITNHEIHqWCM5YJl5moLBmzQBe5Rkq9K0BYNmfuPxNTYenXq3v3kng5sB8C9144CIhXlptYVyGyqu1RpPgezovyaBO5n2g8hfJCmYkhnMYtqijcPOcDnUKcwRB0a9g++VC9GbpUkZJasQ6pHX4HOsn2JjahgVHjqLytkAT7y0qCJfxOmdZR44wFbobBg0aIYjUkc4IqjODf01kMHdBqvbffAmCxcMmIVFdh9cVq/CFFiscIUNk0BDCUX4ic0dQmuyLe4daEHrxTzHFxBeKjRf/95syjzpmidsM2mfsQXu/w7nZk4CtgPYvCCC0Z5jRxqVHT9DXW0RiML1oojDogR0ulqCjgTLybMVZLLqlenLlHW8jmpFUs96bObdevqgHsFkJAuNTGSTrQa1jmjYSa/TJR4xaOP1UUlGqtGqNx6jRzyllmdlgjaH+gmfZmFWGkP6gxYYNEzY+x6ikJ6ilxAn9rlNM6vVK4BoKSCeEquNgzWmoL6QAwtXG0kSDnYuQDiAKImHYiYUHu4vrYRGmqXLOzTh7mRNDdZVNQtFpaHWoIF4850CoENZoCa2zGS++mcCKmLdRdtA6k/2qFSFeofVQQMGMyGJJiIfYme7JSd3z5Ep2jmahHgIN27sL+8C31j3cCE4DxsViwQd7ls8ymItkQnupHxMWPZx8J6ajb6ptk0Tc/wAtos+Yd2MbKvaAxT4oNanyqgRU86n/zZWQ6buaY0er5HaVWZlQkZo3wY0eliDUrdrhVsNzMZykwC0Aw4ApfSEzEzaN5Nxibo2mcvwxPjy9OaPwfXQI6f6cnT+J+mrPn7wHVEPIweLm2QOR8ccac49KLU+dmxpqDOHTLX5uMf+wv48umh4CSyg4e0/yuf0sZLZw0mlvgZl6c8e0CuUNJkq1RpIvYBECgl2IjqTNop4hZptKO7YFh0uorKI7ADieqv7qS0rDN+FdTgLf7n0ra7oCNz7SovYfR94EJHXYXDw/uPh5uADlHSmbpdKOh14+zA9dDcHVsf5ACVOFz/5SEXOJ3j8xI+feTBt1ZO5+bLuOZ/mr2bNDP70cLKplo9csZHhrS+EBU25+kPzCKQQiSIZJGPOALcEN5L4YUuxhJVz5oG1m0cLqVoALdrb5vhNQJy5e3ivLw0wSFTTtiED4+xZm7o1VmgpK0wVuBHgXMAqT0HDjdLyFEuHqC0VMhgNbygU9HvXG13eeKeXt6OTnkFkuic3R2keA4Nuvx2NYP5AIeRytG0+uCgU5YbaddcgRYrk/pUgijEIAuMM9W4fg9vGPhXsNDxTNJY4/btQ810DxxhDVY1OjDIFv4d8fuN4/DsBAlo4QHNobX5TsLrtH3+0+5VD6mqRZQvhh92HqSauDDzDJkQJWQOthJbpsctKeAglEPzcOKCYgxdH4B5ZnTY4F/SfUs0ItnEPK+2EGwTm1H1BZduLWJgEvjSlkIzkKzSOsTX8hwxlKaGu3hmbkDUNGg2VsIMINcqlOJAKiJ7AkOzaAZhzZQoWaRq4d4C02qh3SDdhNaun/w8nQtytMOaAX2Bk7AD/qRWbwc6E2/Ar+qTl+dVWOgmknjQQlfYUJkF9AzX23wgsEQl7MPlDt9PXUOnbJhC+nSAZp+SQ4JOO2Fh7ENXUgE3bKDVnT7LPBNyShyR9TGT35nRnIozTNLiAM4hpBtWaCFLRyU8blDi3hQd2aAHib2hvAEzFzdYdaOQr3Y/9z6J0VdAHHfkBqJNqrvLBA9H38w0juT0zrvA3Bk4KFDg/z/0NqOA49UuaP0x/MpsyDbV3Bw/rMgX7207tCyoTMRVBuclC7JR6e99lKigNB+VRMQeHHuQDR7cdUEagn6s1JKiMsRoDbysWWOTAgBeiZpz4iVemHrqaWATSl7htR2QpQOE5PiXhgp94/G2YupenEHFJ4CM2rFEmUEYDJ271yttahkuXx0ZshkTyK7bz97KdafJcQH7cSpNpPTLzQ639E8Us3oyTmNMetFLIDWdi19zA5P7LxUAHmnL4Gy7CvADrn9j1Zvz1kxoM+5r8NXWxH6pGhXHIDEYqplabb2B5DFBI8I3zdbEDAewK3LEMAy/NPcWZMjCG6ojmQiExyxwfD0aj4YkpKsUlqA/F0aBz/EDM41VxrzDYE8LoDbm5+A9LSeXmql0kEbAFzkrD1xXxHVwzOLwEfvA7X0jv+LHoPanV+nZ/o565w34FmyVMluOr0rI26U89tuXXYIYAbigzfun9PgBUAVD09vp46F1f/jE2aVUojQXL49OuLMGKtQabugGQ8TNMS68N1Kz0Qi4h72ftvgDm4gW8/p8XWtmr2cMNsYCArivX9ndr47q8bnCAKLbD0LMCnlJpUmmtvRBtgvVbIteiEUbY+K+B1d8QWLG8fASlCKKP55vu5IF6fxBrdtLXauuwx/VP1SDTHaCHQVGPoIu/CHX/MaMzEuZYlIUnW+hCxv0VYHbXVQSy4BA2CSRHLKuBmf9d3HI558oxB/4vQBpiV1bY0qvit738wRT1js8H47F3fPn78HrwdeidDs7OhyeGeYhSp3UG+4uLfoaZLAHyFuDDWXQ0K57cJ7FxgU/cQEhmqDjg30Ab+rNyJ1hNwoJBagrPU4AdT+eNAwpTH/DerzIwRgByDBnDdlqNILfOchOZduDcxf5D2L2zqI4U72IF5oxboc13ypQC1RbsRLQIweeiw6BO13buwycusmwdPDxN6+Vbu+fQ+pScJNuYQi8hKYhjPr5fYDJA4bb7szwtVNyjKm0ZqyT6pwB1hCvD51Vo6X0JdjxQ0YGuDZWWkt/raBUQnUSWUfAkHpqQET63e/wRRaof7Cl547jhwQQ+4MjFnimiYCIXr+2fO7UDxZUOm1JrT3xFHhd5/iqISl060jwCAwq4SooB6gof1MSdVBFr3gfLF3c2CyXaRyWVr4JFRRcieZ7GYRvdqgiF1Wlg+WKr9mQBC/QwMGQCq8wnZm1rzCmzA4FzrU9bTqAp37xxygo12IOaYD3wnmJu4KIy9/UejupWjU3QelIbsmeTtJ6B/0FxmzM0GIHBc+UToPkqjo/UYV4IX5iCziA8/SjINbeSkg7xfA2Eec+JEzMBpkMhpfTkp3f1wVmfxi6Ba9URFCx1TZpcP4Dac7yLU2fvIYnQ4BugHPXYE4NRg9RlXz9ARGCS8BEkqpgYvDRAeWrbJeFAchAN4qGrSS6rdgQnE4+QfvRNQd4NmYflMk3Y91UHrOjo8rEruVpLH2YL6qSOjGPeme4BSESAB0dYrJ8iySg2qHxRhm61OxFMAW1yQ8kI7Ffgh5rjA3sNSgq7nVgmbYUp8DDIMusdMaF7LAF9WEJtaPOw9cwbYEwmpbnVaAhXA4FDuTReQhwMo9qC3Q4CiKg6QaQ2oFqKqeu2bVuZiT21HahFdnjyWBW7UiNTp41tA8exL8+LtGB0QP3Y+L+WkQUO+renOYA9q1oTmgAPXEdWTkxZVUkxOp6Aewq4G/WrxI+MIK6uh1eDa4SRUtxNlFyhmJFVETreawG8RMfI2FDfWfsAS1T4rIE45flOtWIVwVTWrbj3ux8/Ec1gyZZmAKF1A6h9+mBrpq5h2zieuctQmcZGwAZljph37eO/pQSIfgmu03OYpwjdsLW0MjswraeHh7mpfuADs93TYRUOViZtntFu9lmlalaR62UMWqlLIOBPnfY+qDM3YUq3p6dnx5RNdTs6++126H29vry9GhsKnZLjBJbGpTjCTt6elXhEM0wiKSvYwPv6ZhKHOZXpjV5R+ijGSFt53Is7xlwv5HJwAmXuwPDkcgXFxJ/asAN6yR2CEdgabetBSWztPO6a1gBUyFaDFqAtPOqr3GUMXxWg5zqBRToHJgnLI926v0kZcpOqJxa+Mp/aLh06KP4+esHNf6WuZJsQ23AxIsSRwobb5kThPnB10c4ZB0zoA8lWBaipjZHOZqu8L7NEhErIuI+C0Gn9tzVhdS5RSgVlg+/Q3c9iF2fj8dnoq/flbDS4/pNhtSEX/4IbD39wLsRoUq6VfmfBh6klBZo3K/cfUQT/zTjFKJJQacvqFMooQLeCyYjTRf0RZSgjm0V0UKXlJVPKxjwqMV665f7BQFsOIHGHOrzPnFkcZaRvYERQdwfwCKN2qMy/KyzV5EkcdcJOV+A/oHvh7bPofLZhIOzd0kM1vFaVQ2NRHkwhE6woiE4F+ixlkz0TlQ4wN2CehXVvB4G4WoWB9vlCXwYn3pfh6OzriH1mTqAcH18C/MD9wiQXEjrSu7K/fmXBYELQCy+BtLtsQUYoWqelSx1Mjh7ofPuBfU9Zhud2pctxMYw6gbT7cwwn05eCZA/TJmRmVd0S04jkSD+7pfT4uNHPPJfDTnjU2cuvJuZ1j3+5PMdEy98H12cDUIxfrsE9Hun8Kbg7eQYGTIwkRLvE0XMsgdd+HU9XJZw7DWYVvvaahegKADpW6cpWx24RcDMuzq+sJHEu0mAVh8gJOKjn4R56Hh4OzkVi94pVhqF2p6rAIFo8B7qVLnw9DsEQwEL9GPs6j5LQz61EALcIeL8Oz28tepI1nz4I4Mm9Ve+Bm5GoOAsQsEc/D3gST7Z2OEkDg2Q4BYwbPofWUUeyOGcekr6xCF/6sQBPWaBLRsCpKEPE6+EabDAlGzLkEJjfgFQjk4GhPaQFJX+5SKInhAFZiKeVfSoFifQDbu5J94ZGU9UlWFcapCqgOSFbZ6XLmwJP0dIZBP7S4jaIgJch6nwgSpyzpYtzPgB78wiFDFxfLuAxxDQrQL+P0AaPjSxbmwspZ3Kw5XQrD1KCZa8+OyRKUQCN8jfp+HB2n0I3FlETTfEykjCrTvEE88uA9Ak0kozxuqrkp53GaH+f7iRB/MIDowvibj1NqNFU7pSraLnZbbrZ21QqLm6sRayB5g5iH/DcfEKvFG2G/VaePbBQAW6QI2Pt7O1hdl6abTz0QzzQxQBJeMusJ/CSN3cSFSE/EZgH/A5+z4bcWis7WhCn0oElB9pkbsACqf0MmnzP9jf3kwO8NDX855DoKHNlzemB8/FADikX4ODdD5Ieu49Lxs20ah3z8tBb1/wxy1b4bwxUkFOT4TielsbkdhUHaLX+l7yqIUlpq4NZbOgxcehLkD/kF7mDKFW7ErUjTfSIFTXvOuhgouJhU6Xcu9n9KqFgizru8BTUe8KERKsjpL2YhVFsKUZ896H9H5/QYdEQB/UEuo7tmor4svqMFss0Cqyltcuo9Nk2o9qKqnQthgI0mn1PE4wVJL5uJps2Gw0tEXSOjhLlYlv8h9RZIEidYGySgLGQTiFmZ2qUBkziovlxQHvPOWRDJ6boR6/wdI3873afRB3+svZE2jN9gTA8qOtuZ56TG423alRap7pOcw61x2haMNU4h1asRJvFAOxh1tHSL/EwktVcIz0UtAiaqzj015gr+pFyS4MwK+/dj4Ku8WAAFuMFrtP+KChaex/hiS9HcvGTPFx4nGcD7gOg2ru0iEAmjzoi8f5K79Cp2HUU1IQdIDxsLpF5Cj4DPwD3rV0prJLTQQjvfGgJWzG1Jz3RmQI532iDW1U3QprvXXqrOvCprD+p+F3BwVRh93uF+Q1BxlRA0DscaXxEhgXJ1SzFsqGXkEAq5Z+JMgkkcZQzOX3jFAG4ydpuC4yfbMj6VFLInQPmf4+wEi3M0s9QcIWqesvlVDd3ZJB9cH5+eazu6BDCJHcTCS8D/didO9nVEjhoFVWhLdrOdwbD9W7Xltn2AW+odv9BBjKlMySXINe7JVx4W4PMR+HixlZhpdoms4RruT4zAjQk2c2A89YqVRYQmo8aIRWkTFiVIIsd5gfv2+3D2cGP/1H3q/SJ7ADZ9tCtamm6a3dL70O/qJqU1QaB2G1Aeqvum47IK4JU11po3nQ4TCERFdhTq5dkrdeteITXWxPym5TcSxaMFewjikbVyWz6JnWazag6+1d0yr6DTkwNZZ6ytc1UqQpKbqSp8kY8UTL9rBnXc9U5RDFpT6c2s0SlSytR8Fd5OnNVNoF2bRJAGOk2AcNjsyxXzd64T9n4gMFNQUcw5F/R6bv/FIFNIuZR5bRSVdHSzsW/T2BI/xEjobwqRvo+TpIfV8fF/z12Yc/Dqthdu65lq7N8mmM1ntwCu/9tFlPwX3fksR8Jh2lP7L5s9c6VDwxKpcRw9hOWoImY4Nv0gDfzkc0ClotHhV2ZMKICqtREJU0Vr9RrA3Acut0DXdMfsP05+PA/pUtqpFXRYL0byBB6fbmnnuRFSkIImC0o9oGpIwBZgjw5oHPusVlnL7Oxao/Cwm6jTF6UU/ZUk1kVsfGLh8LKojgtZWgcPXQJG2REzpxiGjK5H9iOwxzS5raMQjaWcfA9TfVqAOdFdeNQu2VYXy3c/lreD6yIPsGDQzFjLvCRBUCu8PmO2KGoM4uWhbx4g7eWgJK4m4KfvSgQCT+S/GEwzaUomC8PwO1+7j+6WD4xn8xp/8ltBvL6G1m5MTGRhJ8DeJ5nVUffcayMoSQQP0pUwL2QKYH02FAWvFG4BJBmIg8IMpIq+ne1ENetFocIj/PwMM2FHYe5OVsFfu+lavTv7GpgqcNtmOkAs1HGrFYdFZ6/9qOYbqWri+IABE15Hv0k6ByJvQRWOnZz/nKVMo8Y86Xm5gsu59XzXrQcOOzGxjIKrsJf7u3VxNxWugnpUfoQHfPwRbXCfAd9QZevfAzdR/HVWklpxp8ZAED/WLema+aEk7Se6+vwGNyuuqqvuMiTo+vh+PZi6I1/PbsyMeNeP/LRk8NUzty3Tv0yUVFPWirQ7EhH333bPePumORWl9Cezh2a7LGNAvioW3nOPkQrOe0n4QLUxRrmtp5swNHAA70pp2TS3VMNEcnoszrX1G7QytNN4hwqdenCLZERtbEaROWo7b2bpzpRIzegRT2Yy48CVdmqcE3QUqv5PJoRhpY96scAiVboNiZj21spjvdI2JjGqKLZqrWKZzPYykAL/4yt+mWmn3pm2d8524sOPm3QCDKNaX+bNrVpnKr+3+0uyjer28qc/9cIJn9sBI/Nq2W6cu3VeRd8xqTgszF8V2t5xz5NmRyqEnX+64FQYspSmak22eE8sUWFz5o121/PO2C8qxb4WLWhboDUZQYSU80E5uCWiQACzmWWImsFt3m6nTZOtyUukhJFckPWYessevJcX/fUnMznv1F/U2Arst7EUXj2pDMdFeLxEzTCI29qE9Ir78ruYXq9pXUrJkuxpxkeqvOZ5b2vDtRxAG0HtRqcydvomi+KFd+LrbXmBweopgCfoJrHkEmWUlKMZEfakFqJ8uWY77kpY+zNTat64pXjiTOoQHrmm+GcbijpwnG8otxzXqVTCXsBXaqVyL5IFTboKXtc7+nRrvXlf22t/TqxSvtcMBdL+pHFuRmMf/UufzXFFpAB62u+44wEaWVtsW2NhTm4vb48NjmEZ3HnE5M4wpyKD82MhT1ZzziPZbFwq9/9cRAK+SUAm5ll9xlt7E8s+Zbo8VU+6NjeWSnnLJq7S4HWk6NOp93uTZuzbmaoVfhCUqoGPq/eHLAMZiPJvDWePR+18fP+0+AXpLFs/Gpw9wyhDewxDPqgKYosxHQqOYLxv8eXePhGQHSBSIiumNHdWLr9ajWQjO3wPViy4Z53wBNkpLfdsvm7PTKcGstb3oW8B8G4kQhk71/U1dn55Y33dXAz9M5Gx5cXV+fDm6HxUnf0+u5luyc67ibQPKFLPQh3wY2gext8hV26afUFXp4eBuO2D3Lzig/t2rvGzvef6vJ88WyXWXo3NZGJh8TW8xKV6cswtyFgKYq9fCVv19KAAn8jwzXH6bx8xETWeegX0V0EgG1Dm0dJrY5xRe4JawNKeMUlUdIrzn/rFyi0M3v+cTGH8sgkq2vUvxqMxyoHiWezk0XkLxZ5WHGRZIXqpCLTMtX3sxVlxtVc1ZP1Sm7f3EuYL157o0uoNS+ww2irTJ4X3Ekp9Zgjw04fP7HoNnZYjv3a2kr3+Zurhnor60cxqUdR2OtbsMovWgfAr6oH4lN1N4jH+xf9mye3V+dnxwP86aGqe1NmRN3vS/Pbm+OHnle+s1ZyNTQaue49/xqKIpXrUuS2mrKSYukE2/QTKPD+Al8o268nsubFK3zR0Zs14cCexnupcHr2DyAA33K91rMn6myCEp1B9C9BCDiLEl+q0L362QBt+UgZurn/8tDLJw88FfJza5oIpoRQHCNMCbLoScf0JgMreKgoIbZXK6Rmwb8Aa8zpq9w1gnRAnW3corYvfXRfDg5wsoBwnl/7tFglJphxLnfpGXaO9ArunYaqaVufG9QH+GxOf37WfSZz+o1cDo3op1fXdUIHnTFUNFdzgtn+j1J1+ir29tgIV0HzrcAUlOwJSkFvr+re8UxSvhGZArLLp70QoqFXCErUykXiCU3HwMRnlKm7dBu5orSHdn+5bKSKatd5lQu1ky7ab/RT0b6Zc9pc9yx9I+9U70kFKBv96IR5e0oto2I2d7mcLPnwDeybzoVTZ5bi4XBLXdVwJ2bDS1PCg7wozArmV5cHwR3blSGwt6vl0s83bjWSQ6l9dxtLZ7mK13SGtydyKlMHjJmFu+bjeEUZEO+ACEAjBy9aceL6E05fDuion8OYmJ7MEsaju6LMxQyvGNEdQ6iytZ+aaX5ZT76ZL8xlHk4GGDnYR2/8waxar2nxLhkU4Atg3yaJlMIGQShBAEeCnSydetmKfkpw+Fcl+XrnNzaxuVn1RX7ujT9/44fctGM/9RNR3F7/gTjt9+C2orjqIKM6ZuM0IFqZQ385rV/rulHbIwp/jw+xG9dg4rj8Z8vHUAO66kEAPsPQGvm4agp+OVH+BXUyPdJqqi9l3W4iOs29KciZH6G7iu6rn0cArfaKMToKFVfNkSj4C5ry13CU73Kw5bHYPXxXKcQabJtvwzb8fQ1outfCXA+P8TQbHS28kYFp8OMhG5gtRA0o2WP9sg9R4wrjsAzBkZLIyiOgSdhaFQnVSq/UcdluDAFrl0tbVDKlV1SFeDmnCk5vfXq01ZLub5OB89gNYIcc83mBzVTMBVzaeBXISsrRlD/PRwF9wnqyBJUhXYyXP4Lr/GeU4W8vWLJe4K94VHVnV97J8PQcsaVQQQ26K+PyNfhnhSkrNlC/iiRlCzez/gml3jP/CIY1F7jtMYUovTKlcwfNxTg9Gw3Ovf/EOHU1LTbL6p1sB6aOcQK8wP0u6t3c9UPowjz/Vor6nWBnkC9WaEav8A2TgDK8gOb5stgyj47QMzIF5/ihEpe/soGmh2LWwkTfhm6sEzSp3B10Ivin6uSm7OmcRcUUdMwWUVoB/1pa++3WxZ7mmMXnu5lDq8JvCvk7G76jAuxy2r3qV0JanF1QtaA599hR266rF9XT3DmZkiOPqrix8B1+rx4K+tk6z8PTT8+DzjwWJs/s8Za0/h8pDk1/', 'engine08_original.py': 'eNq9XGtT40iy/c6v0PTEXUm0EIbunp0xq4lgGtNDLBgW6H1ch0MhrLKtwZY0kkxjWPa335OZVZJsTDe7EXsnOrBUj6yqrHyczCrNmzdvjk9Pjq6s3n2uimSu0srq/Ni1ymxRjNROls6W1jhKCutYxed5ZVWLNEknVpTG1ihL71QxUelIWdEiTip/6xi9TNc4m0dJak2zWZwtqvLAGqFTEkeV2inVTI2qJEuZTjUtVEnNdkbRLLkpIqkplBUn5W9ZkoJwP7OmigkZuuUoQzduNsrm+aJSsbVIq2RmRSuT/5KkqSqspLRkVBX7W2/evNlK5nlWVFZWbo2zwrq1QNKxz88uwv7ns/D618ve4dGV7dnnF73+L6eHV2vFZ38+XSvBW+/vF5crpW53y8J/Wemr9C4psnRwOwzsPdsMPirvPKtclp71W5mlnjWNyik44FkF+JLNPeshycfJTHlY5iSPihJPVRGN1E00usUjtmtrXGRzK48q6mhpuhd4NWOki3m+tKLSSnNTlIM6CvAvj01ZlRWjqRDjR0MqTVuFfpqa8vEi5R2MZkTnWBqVtzMVFak/V1WRjErTtshGYbQYhbxlWAukJpqoMC/UKClBQyq2ts5617+eH10FAxvCdng3OTu9AGPxfFFk9/XLYRzN65d/ZJOEXoZbV70ede147/e9d+87f/T29t95+539H4Zb31ssl5hTNMOaIQ2Y4DSLPWtRkjCT/ERFdDNT1s0inqjKcspoDrmaJaQOZaVybFGRLdIYv2U+S6rS9Yms6MSsCEuR5GhBxG6SlGUYDEgrUGA2QRTVPd5i60YtMxZ7ZYFbGKOA4O5AsSDBqorCMuj4nc6ev/XxsH90cnR43bsKHresFlO6gzgZVY7ML5wVwb1XqgJspec9v+PNF6DhYTpEas+1SMTvScQHRLrzzuMR+K88d/aGQ0/GMMx+Noj0eTbQ/QvjNGPQn4a+2b9v0o9ay6iEduRVTN1paLueo4cwzx/WyvfcemwjLv/PYz9tbcVqDHM2UWXlRG63UNWiSI2+++U02v/wg5Pmfgl9od+ojIoiWjqRF1fLXAUogiz98N51/Sq7WVaqdPA4VfeapssDwAqGZEmc3Mvcbu5DqrE+f34bJ4UjL2VwXSyUp+6TsgqzW35zD3L/S5FUKqwgog5R8GOYjdLJvCSN0SvY9yDB2ZcwjdLgGFqkPIwXLWZVUFaFK6PPsigO51m8mCmHDBJZP20B5IeWuoCF3oIOBatFfpmrUUhGJCSDF86yEauQY49nSVyG6j7v7IfYBn760fYwrAziHszXackchBrRdcrcPYCZ1RXloMz9FAo+DOYHeKR5q8JX95iBnv7cPdBbNJetK5WKQ7DAKWlVYp59KnSwLSh0D7BDm4rFbs6jdBHNwg01papCmOiQnGAUl86eC6aNtbkdLeLIT8owuouSGdknx+22alpUeW6assyYPXJ4R/VksJy7yFt649yLPWpPq/jeOlKVKuZJCllIRtYEFi4ne3gDl68iLF9ZZTJJ+akkYwkDpX1qNSUXzG7Wwhjk1rFb/pYQKYPHpy1WmiRmc3AXkSe8VUsSF2ecD1A+dA+kMfFAS5ODJt5g6PpRnsNU8orQFGvCpELx/MHjQ3cwZOoPFs+igFN3QMSJB3fR0HX12EQLskNt9EAQ8TkUh+aiadF0YsymHHSGQ4wCplyZJcOUL2YwyaQqVjQqsrLU3cquULTmC1SlGTw519a4ZreFZMAU2tGZSvUcMRom+fNe10zCDkPuH4Y22tYLbfNFStqscfTyiDd5MroNBsMDjIofWb3uwUtusamhrpnB3FhjF1cxdlkEBL9Yk8TqrJksmsVb+5/2W2IkCxbeREVdgJ5RFisyVMZKDbo/Dj37JpnYxGvLoYkTcxb/sx8EHUvBsAAmztCVXaVD69PLaxlFvRQqdtftI/PgeVsiut6UFY02j7aGiVlggymgHt0iSjCjS8KVc9Uriqxw7N7ZxfU/wuvP/V54fhl+PDw9+eXy8PrkvG8LRdpkSDgRHLp/0G+gNtxMjgmBSnh80v/Uu7y4POlfh+d/7V2eHl60KC41we+Cx46390QzlWKmrEu/PsBHwNir8Ozk6gojadLkchZzTQZb4P7JedehCmBWACHAVsfm+CAEuIUFvjo7/3PPdgNCsbJdewAPm1fWYk34S69/8qkfXp+fg8ThKVYL3nXCvfCid/mx17+m6WiLS+v0MB0xY4C7SRpmKWyydw8Lpl02BIPtmKeR3Gg88TRAi9VdMlLyBj8yIQFvzDeJ6JYFM69mwdwHFnDuSZZzNdiDQSorhCfB4233zo8BxUZTx/VH+YL+zmgOggRuvTvSEibic5eQ4YRrdOqJRqBQajEnYmKyHxR0PJwlt8q5W6HDFFpdS4XIKv5POiYPqmwL/4AEOZE+CfVouDfU2jCG96vcgy8qmUwBDpjELv9lwQCvRlFORsCZ+x+PPw1sTULYizobbLuJqtE0pF7tlk2pTaZVd2Q0HQzm0b2z51FrzHekkpkDP+TwfO9dbH/u7jb9XdfdbhEmcDALVZ6NpiVIy/Lu19YHswFMP8eSsAz4+8VczfT+5c3ucRvygmQJ4ZIrEGZDF3TgciACex793QcW/Alw7qefIKbpJGicvTbQIUpZuN6+Bzj8I5Yr/hczocYImn9fKF5qlgLZKHgY5QzQyR9NM4irwwJNDCQ2vPvxvcesIPPuFSqfIeYT3KVXK3a6tVy3HvI+ELFho83Rn3M/qGczJADpiI64B1K81D2iEgAwhcWEPWg6aEHRRElc3u1rJQs0HfE4BYIaTAqMmShHlJGdC4tpSf5pBifHT9rljBLjcRTtT0FMadYk0TNv9gZV5XLGbmFLBfnRVDYL9dmMOOx0shw2Cpwu9JL43aeoxJFubZnwEAvAttRiPytYli2reEkIGISxIwQ/tt91Om9HyfYff5JOahwM8pZpaRkVFsrnEyBWcT3pTcPetjINRslQZxpugmKzRDW6tEGwTMmfWhrH5O43ytLNqgwtN4nPzevEpr0dbOfCSRExlgurLEzBHQlQuCVJT3DsU3RdLDViAhOKLF+GhEjJ2CMyl1107mFGlkvpCX8mfgJ+qx3hcqXQpT9vO/6HbZ6qw3s+X2Czt8l+OPnO765f/r5ADIV9Y8sou+b9TpvykOSbhAf77dYzIGAh3EjKMTA3ZJ0GNb7zmMYF+r6ghJf2oP3zPlDByTX89+kJvGR4en51ZTfc8CkP9CUqYsc9aPhIQiGiLppnEOMr3ZqsY6Nbc7UGG4rCK54JNYEJpWGjuwk5L2KSNPjibkeDW4HsX7zIMEz7HE9m6epJ1I7tiVC6mlUREQNNUNjhCkNqQ1OITDEXn+bQ+A9+nC04blrbPD3fJorg7j5CmQWH1tvb/gdaSwogqkHB01ZLkCRd+EKaSpSx6YrJo4DQUtdoczN3EcJJwBPA0g4McqBcIXug7VbJW2dvhwvd7ckm0a6TKwIiDI397frdUNgHhZopTIon2PSrn3Y29djW4SuCJadp2dQLSURTlEhDQDBLRkkFLt8kkc65pXeSGmN+3GTV1KTTZD2lzxTm06C1/F2z/O1tB+b17R6c9t00aGZgGuw3DUjIo3kekhnsyLTqreF16oe3Lc2vE0FkAObTXecOS/29gC60W1URGQiXXZzK12XPaY9SS677GolspKMWRG1AKHgx1qMelOOW+u3nPfXDV2zK0Qlii09kTK56l3gMr657HGjIgEE9a5S08NDbYH+bw6LGPW8LwNp+j5bzDFPKig1+Wte85KlNdctXK2ghiyQZdZN41p7B7QowYZstvH6FR9CDOBoguZ7GPYbpLzC4Gep1Rvrw89FJy0bXEYgxl7xuLg1YNk0wM4tKGSacqyjVy8JE6M0RiwtvLdmWUAKiFheaR0OvZQqbR5PcXOR0BCO1tcx4o2w+B0iVpJtGwK3dd5vwjPfNa9UxiCcRbmMSd1sDQB3FmSMeKMUEY9whPqPth/FDQbAh8Vm3q4N2ZgtlQA9uAx03oAg7QJRYNtGHPVDVsJCkORrDHVPlYAeeA/t8K5Erl+wNPU4IjFtLrLzVLUDDn4PKJNbkcEPOLEpnWXnliDOdEqc3YH+JHl+Lyw8/fwwve3/5fHLZuwp/Ob/+VQL03lUrFmaJiRZFNtJLWjlK0YOTOOeFafHC6UrdNqfu47zofHiZpIfojNpQWvyDaxYu2a3whjI5RfallBziFXGjmi3bh25dkpI0BjABIIkpEKuSaGZdnn+0sGoLNl8fI3INidwBKqj6plDRbWlViSL7P8ng1waFRBcMfjEscbqAiYYcLhD+we/R+c0MmMvmQ5i2BqOdjBQ2y4bJHtapHxph4+70z8O/Hp6eHIX1IUxrW3RSiTp7lNSEf7mJI6voOjubR/Ra5bydKEJJfSJKbmTQGQqfi0UajklX5t6XrLj1SG/ayY7SZDrKeXbLOnQPGhHpCBlZipIPP/7acw/uYY5hLtIS/Js73Ghg31PAvgz029IeHsTmJcbLOG/RuTw/v95dzcBjBarICyhg6af5kphSFd5d5MF1zH1ONYcJVjUi3fBiyjjX86e2i1RRcif4RnrapAsQLFbFIMYfbPRDK+s794/Ozw5P+le0kw/fBTQItpXSkzBdrDdsKQOdeqwK12sS1ab0LnI92Za6HSZI+Yc6fWsqOIMIc1fVLRWZjGxRBbRLuyZNVh+jd34ML1WJsLC0D6CLWRGXAVrv2vqlLv32+cyWpQMnm/L+9lcPCMS22YD3toTlDWDV4kOIE/ilDJrzxYFUDY1asGQJpeeNBt19ikrlOLUVzBthpmTcWkyP0QQUk7aM7ce5T14HTIZ5cGjr3KcwfKSdp18ZB0+jx5pot7MfP1F6nM+6A8263bHN00AnUH7y6ejKlixe+LV2eWUfjBbwh+utpFC3GpV3PCKYQs183hREB2xmmkFWyxuydbm+BRDwwRqpVslHVz6d+MiRWxMlZpR2dYx1c59ZN10PuDumLHpA2Yl2hWigqAFVy1O7hRgPqpMnruMUeCwzwrKdZhVu3VCHy7LvBtNk7gFbA8eGG/t81guvL08OT20yi/Cii3KqTxgxYQCnhWIavPpFOkvSW2eelBQQNGeRLca+1KTF4xea0DBVMgrIovvQyHE4whoIB+jeIrdYzFKWJbCGYJAnGcrNiedvZZ2ZtmymAAQYRXhiskchZxwF94IiW0INeeFGmysxC8A9c7RGjfjITWIhVQVr6GNALYaevK6kGtpuEJ0G9ooHfMHnGSR7/bl/0v8UnvWwmx91wiETq6rv8wS1VQ9aHAkMY4zaBi2rwCJL8uqJdLdE2xPRsra3H42btN/edu9W0u2qqhMRmutaML0696Gzy3JjYxOm5Q3Wg7UBtThmA6QZkQe8l4QQATY2Y294cUJKG5A1pbcf6jiyVWO3fJgIi5mOEJdIttwkuDsQaK+t31q5ZXt0PA74p0Ws0SOoaOwfwcUfU25KdJtSeGuazmf99zrNfNDcJaCqTAbRqs46Hp7/WdTczgHqIFWyKc+FzfvgenYd45hmr+IueraMCE1B3Y9UXlk9/uF7YqWluv8tAR3DrarY9hQpCNxEXjjYsBf3YJ1nByv8Oj48Oe0dCc8MqbXVrdtWiUujdOlUDeL9ru0TSJj4Xop03azYJ/2P52cXpz1KJNbuvAs0C6gJwcED4jdlyWo1JVsicLo8F7SRvx4HlWAy5qHa3nOalSolD/4a/y7eeoukSpEu/6dk0vzBrtMXogaMX5/7/lfDDln34E0tHG+GDEAIOEDl8/p+SsBAy9OpzJBMdb2b5V37LO4lPwBcadzA81iXl/UaKpX6KpFCgCidwZjAOijvBvWxr8FwVVTQ9TdzUay+vdX5MOxqJKQDbabh/olOf/XVN+oZyEUq0Uc96Er2Q5pxbCmPtaolabkYj5MR5xBuVJpM0rAFw213FUOMFhXheSb0PLkgtKF/cAU/w5BXdEoBZ6nE6cuBppP/wUFxsOfqFJyL6Cddqeo0VZWucf6Vu88qx88qG6Jb/zYrKAG2wL7C8NRro0WYFFCLLdxd+GD8SFRV0eg2hPRHs1lQ5bty1Frlb8epRDDcCUvVNXh6W6GqzhisdsqpV44yNKReHjhRpR6WDHpsB25++3csryh3Y4C1rrWD4bqRWOS6hYDeoay0HU7XTdZi7Geh+HrDFT/FvNF3V58DLYJZlfbWqTgrPjmjyBLvwGHyygHkiuwG5j4JtaMIkttxq0bQRYCaixhaemS8DR5nI7xpOPWsDutbh0mm9Vo55Sa0vdC/Mgm2OKG5Rc0JVux2qDWfw9ZaXFdk1Fzjlht5Ot7m9szdUCwW3yCqfH5huaKsIODMQ0jeDnRLuvNDrkI2io7WKRNJz4h9EayKhkuJRt8gCOABik7LILbDeyZDNpiX2K6oadKOrHQxtO820daCxbMjiK/fmxkSYKfl1WBB+1EPesTlghn+dtLv9y4ZZK0ngLyWrnib1MfWtoXVAOyPJmlGV/wM9sJI0ICWtNtAYUZn3q+iErlMOJkUZFuVU2ekJBIq65xUnYx6dUZkNRuyZYnSB4N2mLwSIzfn5OjkT2bZjVNjhW1BE5TaUxiQJJTVjKfqbsvtO2Ue9aTlDhRrJ5NxvwtM5004amz3/n7R+3jdO7IeTbsnSzbqyrM+nV9bjy1iTwShIohtITfKHikdqW2k5P9oRvIkEzKXWTjXKVSeEH6bsbYsxLpyk6JOvkg0y09sJChDWt8BXOOVhkGGVV1tu1/k98EqHH1cC8jQ1wRkklWnsJPPRbWh9tayEU+sY/+0Hk2DbvuqMclkbeEpwxoCKEs6zH1yNy5IZ2u2OVPD6/keDUazRazomv8OS7s1yyYl91aIxZatPJVuS4noQv3GLDYInA6Ux0E7I5K7B/HYB0rBdjodTzOTLivbXk5n7nNf36eo5wUGMGzhHap9/9hcl2k2mWY+xUCrbnRFWtidNiKjfWpLcp47VjIKa2Xad66619YmtLq8sBOecX1oqh/RSPs/KcMTF7U9INe04ZxMRPtD6YcnWsYGt/Zqj7bRmW01YKEFBF7AANvbxbppdDU4zkT6Brb2ivawy/podpZ28J9ZzqDdqOIK4vsPN5antKJJfOi8EtPzROqYng1r7ajZbyM2ySkZywvOOH1PSrMS86/R1JHeKlFBoBTnhFO4vqK+7xJqzdlMVa69SdBfeoA85PelFYvVyiCstl9gfkP+nIimrHXoW1Ou+b5KcJzcgw/CQsnebyQlGr/O1AOC49ijeDyAAdAYvQ3SyeVwE58vhd8snXrzak8M8cJORmXYHnEAp9sG69Rcw3M8ij/22qkUKswLKtyoDkM5rlxdu8wkBJqc0yF5rMpRkeQUJW1iwffWMfFKBNjc81fRaGq+49tlHGIRGijiGUCZlY0FGR5oJWvffbeY81aVZb4YPWfqla4nl+dB+aXNaxhpFMXTCtJcV+co1RYL5NnN2Q5xkSdCbGpZHGGuXQeyPMbgduin+sAW3n9vY/rk6uL05JouTp8dXn/81bLfTt/alr7y7vLjLVinJaO+3LCa4DQ+PBS8VtZO3TNtmqoWhiDjpaVXNE++PtPouaWMbVRd420TDaUreFwa0QmOyCb3r8U55OPysM5HS2sNbzX0DQmXx7qGk0Za+/my7WtTVTQFQAYV2H+jix5j+kaPPvLcwX6bBDgdTJYKEQj6jVXB35ompdWSYs4+yCcNOzNYfvrA07qW4+Xa75WUVbfU74toZpl8r/6gb1d/3kfgRRp8gSryt5V0rSaaTbIC05snI0uWcp9US5+gXRvAw2gt0lCWLfBKR/No95Dkcsz0LVTsoyVwMF970V9++v+b5Mf4dQwRz/5ieyYgou2pG55chEe949NDwNO6wQyQZxZ8cClL+mA0hzUPc/YLQVHb+sIaHXPRYSKRc9zug3wT5uQeocIZJ1bCKuMIgP2PxCn0XcCPockvmmV7NuZje/WsV/Kc+hTdVOprFXxBFxPJA/Oxq39YTBbEogt6o6Ob3I9iSKguduydHZqNXX+MZu/eIlSZqV0qJhOwqYtKJzDsnf1WN97OZw3JrNmeXKcFqLbBUjqkhSXBZpMxNlERPcN8DxuC9LppbI6SbC8S1YKekB+swJbNM+DoZWWem2hSXPOtRjriWVtzFPBHgvC0aAscf0B8C+jzYSfyeaO55DUn1K2PnFe+GKE7BPaQo3smyCFX5DMjui91ks9M5FtpS/bLzMrsnit0zKscVr+gYfthj5v5+dKur35Iz+aQVt/wgvD3s+qYbIOJ+vQnM3puvf6nk34PESD35yBvHrS/fZQKFPPuBZHPv3I3woGo6JlzqUzbqW8zDLp7wxaDpLqudeVLFsiifOonF8NJnSP+6rA9iCFDxXqUQec5bf5smpYg8oHZmuuX6/PV5dJNf6pdX3uSMHuHHpvZmo+UTLTN1bqn4fdf6cZtfUwhV25+PT89Ov98TZ9XSWvbSA1ppISYjS6yqnWbG3zm/svqsbwjovH1uyx1FzGI2sIxNfPdVHhx2bs4vOyF/ePw81/Cq+vL3uGZvXLUzTn63Nyr5Vu1K5SOzz/3j2xPT3PVNK6tUtuZ/8oKN3+y1V7q5ed+vdzjk8sr/krL2njXiEPvnrnmW46mah7BX8pdNGuyAFhsfQdZ3z3iAwQjJs3Vuab+qzfojLyc9C8gLTyR5uJlGew/E/Z3He0Cp5b+P1OU9Y1s+UiStKXbPuo8hixilw8vr20P2HX9wO75ZS2vXL+mpafwfHfX/YfbfUWeraYGYmFInzGHYUCfkJIDDUO7K4506/8AlcB74w==', 'engine09_retune.py': 'eNrFXGtv40ay/e5fwUxwl6RN07Ink03kZQBnLE+M9Wttzz6uIBC02JIYUyRDUh4/1vvb76mqbpKS5Yl3gcUNBhbZj+ru6nqcqm7m3bt3RyfHh1fW4L5QZTJXWW31fuhbVb4ox2o7z9IHaxIlpXWk4vOitupFlmRTK8pia5xnd6qcqmysrGgRJ7W/cYRepmucz6Mks2Z5GueLutq3xuiUxFGttiuVqnGd5BnTqWelqqjZ9jhKk5sykppSWXFS/ZonGQif5dZMMSFDtxrn6MbNxvm8WNQqthZZnaRWtDT5L0mWqdJKKktGVbG/8e7du41kXuRlbeXVxiQvrVsLJB37/PQiPPt8Gl7/cjk4OLyyPfv8YnD288nB1Urx6Z9PVkrwNvj7xeVSqdvfsPBfXvkqu0vKPBvejgJ71zaDj6s7z6oeKs/6tcozz5pF1Qwc8KwSfMnnnvWYFJMkVR6WOS2issJTXUZjdRONb/GI7dqYlPncKqKaOlqa7gVezRjZYl48WFFlZYUpKkAdBfhXxKaszsvxTIjxoyGVZZ1CP8tM+WSR8Q5GKdE5kkbVbaqiMvPnqi6TcWXalvk4jBbjkLcMa4HURFMVFqUaJxVoSMXGxung+pfzw6tgaEPYLsr8/vTkwh5tXA0GVNjzvtvz3n/X+6O3u/fe2+vtfT/a+NZiicNoUYrVYJ8x9CyPPWtRkZiSZERldJMq62YRT1VtOVU0h8SkCQl6VasCzC/zRRbjtyrSpK5cn8iKtKdlWImMRgsidpNkLJ1YWlaDAjMAQqbu8RZbN+ohZ4FWFviAMUqI5DZUBrKp6iisgp7f6+36Gx8Pzg6PDw+uB1fB04ZFyz24m9Jq+8M4GdeOzC9My+Deq1QJhtHzrt/z5gvQ8DAdIrXrWiS89yS8QyLde+/xCPxXnnu7o5EnYxiWvhgkLV+OMl+sjJKWzTCa/ojL54vl8na8gziarx1P2rZDRp1l1TJa5NVM1mnpuh69dJ8/rJTvus3Y/8inyf/D2M8bG7GawHBNVVU7kdsvVb0oM6PZfjWL9j5872SFX0Ez6DeqorKMHpzIi+uHQgUogmx9/53r+nV+81CrysHjTN1rmi4PAHsXks1wCi93+4UPKcf6/PltnJSOvFTBdblQnrpPqjrMb/nN3S/8L2VSq7CGyDpEwY9hICon95IsRq9gz4NE51/CLMqCI2iV8jBetEjroKpLV0ZP8ygO53m8SJVDpofsnNZ1+aGlLmCLN6BTwXKRXxVqHJK5CMm0hWk+ZpVy7EmaxFWo7oveXoht4KcfbA/DyiDu/nyVlsxBqBFdpyrcfRhUXVENq8LPoPCjYL6PR5q3Kn11jxno6c/dfb1Fc9m6Sqk4BAucilYlhtinQgfbgkJ3Hzu0rlgs5DzKFlEarqmpVB3CGIfk7qK4cnZdMG2iDet4EUd+UoXRXZSkZK8ct9+p6VDluWnKMmP2veEd1ZMBc+4i78GbFF7sUXtaxbfWoapVOU8yyEIytqaweAXZxxs4dxVh+cqqkmnGTxUZTxgs7T3rGTlbdqgWxiAHjt3yN4RIFTw9b7DSJDGbobuIfN6teiBxcSbFEOUjd18aEw+0NDlo4g1Hrh8VBUwnrwhNsSZMKhQfHzw99odiZB4tnkUJ9+2AiBMP76KR6+qxiRZkh9rogSDicygOzUXTounEmE017I1GGAVMuTJLhmlfpDDRpCpWNC7zqtLdqr5QhJVDVZbDZ3Ntg2B2OpgFTKEdTVWm54jRMMmfdvtmEnYYcv8wtNG2WWiXL1LSZY2jl0e8KZLxbTAc7WNU/MjqdQ9ecodNLXXNDObGCru4ilHKIiCgxZokVmfFZNEstux/2lvESBYsvImKuoA34zxWZKiMlRr2fxh59k0ytYnXlkMTJ+Ys/mcvCHqWgmEBIEzRlV2nQ+vTy+sYRb0UKnZX7SPz4GVbIrralBWNNo+2holZYIMpoB79Mkowo0tCkHM1KMu8dOzB6cX1P8Lrz2eD8Pwy/Hhwcvzz5cH18fmZLRRpkyHhRHDk/kG/gdpoPTkmBCrh0fHZp8HlxeXx2XV4/tfB5cnBRYfigyb4TfDU83afaaZSzJR16dcH+AjAehWeHl9dYSRNmlzOYq7JYAvcPznve1QBdApgBIDq2BwJhICxsMBXp+d/HthuQHhVtmsXYGL9yjqsCX8enB1/Oguvz89B4uAEqwXveuFueDG4/Dg4u6bpaItL6/QwHTFjALZJFuYZbLJ3DwumXTYEg+2Yp5HdeDL1NGCL1V0yVvIGPzIlAW/NN4nohgUzr9Jg7gMLOPcky4Ua7sIgVTUCkeDptn/nx4Bm45nj+uNiQX9TmoMggVvvjrSEifjcJWQ44RqdeqYRKGhazImYmOxHBR0P0+RWOXdLdJhCp2ulEEPF/0nH5FFVXeEfkiAn0iehHi33RlobJvB+tbv/RSXTGcABk9jhvywY4NU4KsgIOHP/49Gnoa1JCHtRZ4NtN1E9noXUq9uyLbXJtOqOjK6D4Ty6d3Y9ao35jlWSOvBDDs/33sX2F+5O29913c0OYQIHaaiKfDyrQFqWd7+yPpgNYPw5loRlwN8v5irV+1e0u8dtyAuSJYRLrkGYDV3Qg8uBCOx69HcPWPBHwLkff4SYZtOgdfbaQIcoZeHa+g7g8I9YrvhfzIQaIzz+baF4qXkGZKPgYZQzRCd/PMshrg4LNDGQ2PD+h+88ZgWZd69URYroTnCXXq3Y6c5y3WbI+0DEho02x3nO/bCZzYgApCM64u5L8YPuEVUAgBksJuxB20ELiiZK4vJ+TytZoOmIxykR5GBSYMxUOaKM7FxYTCvyTymcHD9plzNOjMdRtD8lMaVdk8TJvNlrVJXLGbuFHRXkR1PZLtRnM+Kw08kL2ChwutRL4nefohJHunVlwkMsANvSiH1asixbVvmaEDAIY0cIfmy+7/W2xsnmH3+UTmoSDIuOaekYFRbKlxMgVnE96U3L3q4yDcfJSOcUboJyvUS1urRGsEzJnzoax+Tu18rSzbIMPawTn5u3iU13O9jOhdMyYiwX1nmYgTsSoHBLkp7gyKdou3zQiAlMKPPiISRESsYekbrsonMPM/LwID3hz8RPwG91I16uFLr0Z6vnf9jkqTq85/MFNnuT7IdTbP/m+tVvC8RQ2De2jLJr3m+0KY9JsU54sN9uMwMCFsKNpJoAc0PWaVDjO49oXKDvC0ptaQ96dn4GVHB8Df99cgwvGZ6cX13ZLTd8yvh8icrYcfdbPpJQiKiL5hnE+Ea3JutY69ZcrcGGovCKZ0JNYEJp2OhuSs6LmCQNvrib0fBWIPsXLzIM0z7Hk1m6ehKNY3smlK7SOiJioAkK21xhSK1pCpEp5+LTHBr/0Y/zBcdNK5un59tGEdzdRyiz4NB6c9P/QGvJAEQ1KHje6AiSJAbbDI3XlStRxrYrJo8CQkt9o83t3EUIpwFPAEvbN8iBsoLsgTY7JVvO7jYXupvTdaLdJFcERBgae5vNu6GwBwoNU5gUT7Dt1zxtr+uxqcNXBEtO27KtF5KIpiixhoAgTcZJDS7fJJHOwWV3kipjftzk9cyk12Q9lc8U5rOgs/wds/zNTQfmdWsXTvtuFrQzMA322gYk5NG8CMkM9mRazdbwOvXDVkfzm0QQGYD5bMe5w1J/K6EL3VZ1RAbCZRenilXZc7qjNJLrvkUiW+loBFEbEApejPVoBuW4pXn7aVd9/xWbcniM2OITGZOrwSUew6vrAQcaMmDQzBolHTy0FextcljUuudNAVib36HlPMeU8nKNn9Y1r3lqU93x1QpayCJJRt2kmLVncPsCTNhmC6/f4BH0II4GSK6ncY9h+isMbod6m5E++Hx43LHRTQRizCWvm0sDlk0TzKRRJcOEcxVlelmYCL05YnHhrSXbEkpA1OFC+2jodUxh+2iSm4uCDluktpEZb5zP5wCpknTTCLiz+24bnvG+eZ06BvEkwl1M4m5qAKijOHOYA6WYYow7xGe0/TB+KAjWJD6bdk3QzmyhDOj+baDjBhRhB4gSyyb6sAeqWxaSNEcTuGOqHG7Dc2CfbyVy5ZLdkccJgUlnibW3vAVo+FNQm8SaHGPI6UTlPNReNeZMp8TpLdh/QI+vxeUHnz+Gl4O/fD6+HFyFP59f/yIB+uCqEwuzxESLMh/rJS0dmujBSZyL0rR45RylaVtQ90lR9j68TtJDdEZtKC3+wTULl+xWeEOZnDL/UkkO8Yq4UacP3eO1PklJFgOYAJDEFIjVSZRal+cfLazags3XB4ZcQyK3jwqqvilVdFtZdaLI/k9z+LVhKdEFg18MS5wuYaIhhwuEf/B7dJ6TAnPZfCjT1WC0k5HCdtkw2aMm9UMjrN2ds/Pwrwcnx4dhcyjT2RadVKLOHiU14V9u4sgq+872+hG9TjlvJ4pQ0px9khsZ9kbC53KRhRPSlbn3JS9vPdKbbrKjMpmOap7fsg7dg0ZEOkJGlqLkg4+/DNz9e5hjmIusAv/mDjca2vcUsD8E+u3BHu3H5iXGy6To0Lk8P7/eWc7AYwWqLEooYOVnxQMxpS69u8iD65j7nGoOE6xqTLrhxZRxbuZPbReZouRO8DvpaZMuQLBYl8MYf7DRj52s79w/PD89OD67op18/CagQbCtlJ6E6WK9YUsZ6NRjXbpem6g2pXeR68m2NO0wQco/NOlbU8EZRJi7ummpyGTkizqgXdoxabLmwLz3Y6gxYXipiKy9D5XMy7gK0GnH1i9N6e8f02xYOn6yKf1vf/WcQEycDZRvS3Te4lYtRQQ8AWOqoD12HEoVxZxyeNoJ1Y2oUqptJWIHEYG8pAsT+2nuk08BC6H8Dm2M+xyGT7Sv9Ctj4Gn81BDt9/biZ0p+85l1oDmyM7F5GugEys8+HUzZkqMLv9auqO398QLebrWVFOpW4+qORwQXqZnPvAb2ZyPSDrJc3pJtyvVpfsDHZqQ4FR9M+XSeIwdqbQyYU1LVMbbLfWG7dD3A7IRy5AHlHroVol8i5FQtT90WYhqoTp64jhPcscwIy3baVbhNQx0My74bxJK7+6zrjg0n9fl0EF5fHh+c2GT04CMX1UyfH2LCgEULxTR49YssTbJbZ55UBPfbk8YOY19r0uHxK01omDoZB2SvfejbJBxjDeTldW+RWyzmQZYloIVAjif5x/Vp5d/LKTNt2Uxx/zB58LNkbULOJwqqBUW2cxrQwkm2V1sWAHPm4Iwa8YGaRDqqDlawxZBajDx5XUokdJ0cOg3tJf/2ikczOPX689nx2afwdIDd/KjTCbnYTH0vJ2hsdtDhSGAYY9Q26FgFFlmSV0+kuyPanoiWtbn5ZJygvXXbv1tKpqu6STNormvB9JrMhs4dy/2MdYiVN1gP1oXL4nYNTGa8HfBeEv4DlFiPrOGjCQetwc2UvH5sosROjd3xUCIsZjpCXOLUap3gbkOgva5+a+WW7dHRNsCdFrFWj6CisX8IB35EmSfRbUrQrWg6n+Tf6yTyfntTgKpyGUSrOut4eP5nUXO7AGSDVMmmvBQ274Pr2U0EY5q9ibvo2TEiNAV1P1ZFbQ34h+97VZbq/7cEdAJvqWLbU6QgcBNF6WDDXt2DVZ7tL/Hr6OD4ZHAoPDOkVla3alsl6oyyB6du8ew3XZ9AwsS3TqTresU+Pvt4fnpxMqA0YePK+8CqAJIQHDwgOlOWrFZTsiW+pktwQRfX63FQCSZjHqrrPWd5pTLy4G/x7+KtN0iqFOnyf0omKx5tPpe/VNuUGVhgM/kulY43zP09c6UPQqOglQ962X6T2BAVYmT7Eje8GbLIKMN3jWC9GzF4IdABc1E0N1cCxl6eTnKGZOYbSajuuqd0r/kQIE7jQl5Gwbyst1Cp1VeJEJfSmk9nTMgdVHfD5kDY4L86Kumi3PKdLvrzYdTXKEqH4EzD/ROdC+tLctQzkCtWost60KW8iDTjqFMeGzVNsmoxmSRjzi7cqCyZZmEHoNvuMv4YL2pC+kzoZdpBaEN34UZ+ghOo6fwCjlYJYJCjTqf4g4PiYNfVyTkXcVG2VNVrq2pd4/yrcF9UTl5UtkQ3/m1WGAWA0WrWRoswyaEOW7i78MH4oKiuo/FtCOmP0jSoix05hK2LrUkmsQ13wlJ1DZ62alQ1uYTlTgX1KlCGhtTLAyfqzMOSQY9tyM2v/47VFsPQGm+ta90wuWkk1rxpIYB5JCvtBtpNk5Xo+0WQvtpwyccxb/T91ZcgjSBarT19Jo6Oz9Qo5sQ7MJy8cmi5JLuBuWlC7Si25HbcqhV0EaD2ioaWHhlvjbdaC41aTr2ow/pWIZZpvVJOWQttL/SvTIItTmgsMadesduh1nyOZBtxXZJRc5Vb7urpSJzbM3dDsVh8t6j2+YXlivKFgEKPIXlK0K3oNhC5GdkoOnSnHCU9IxxGJCwaLiUauYMgQAsoOh2D2A38mQzZYF5it6KhSTuy1MXQvltHWwsWz47CA/3ezpDAPi2vARraB3vQIy4XvPG347OzwSUDtNXUkNfRFW+d+tjatrAagP3RNMvp8p/BbRgJGtCRdhsIzujMd8uIRq4ZTqcl2VblNLkqiaKqJlvVpKn+3VzJcp5kwxLdD4bdSHspzG4P0tHJn6b5jdPAjU0BJJT7UxiXBJW1jWfsbsr1PGUe9dzlkhQrKZNxvwlM53VQbGIP/n4x+Hg9OLSeTLtnS/bryrM+nV9bTx1iz4TCIkhvKVfOnihfqU2lJAhpRvIkEzK3XTgZKlSeEcGbsTYshMty1aLJ30hAzE9sKyiF2lwSXOGVRkOGVX1twl/l9/4yon1aienQ18R0knanyJUPTrW99lYSGs+sav+0nkyDfvcuMolmY+gpBRsCa0uizH121y5IJ3w2OdnD6/kWDcbpIlb0XcA2C72V5tOKewtwbGMG3ZYy1aX6VWCmXjKdOE+CblKlcPfjiQ+wgu10ep5mJt1mtr2CDuXnvr5w0cwLDGD0wjvUQICJuU/TbjLNfIaBlr3pkrSwV21FRrvWjuS89K9kG1bKtAtd9rKdTeh0eWUnPOMB0VQ/opF2g1JG+j1acYRc00V1MhHtFqUfnmgZa7zbmx3bWp+20WKGDh54BQpsbparFtLVGDkX6Rva2jnaoz7ro9lZ2sF/5gVjd6OKS8DvP9xYntKSJvGp9FJagCfSpAXYsDb+mt03QpSC8rm84Jzz+6Q0S2mDFZo6WFwmKkCUwp1wBg9YNhdiQq0566nKvTjJG1QekA+5f2nFYrU0CKvtF5jfkL8soilrHfq9KTd8XyY4Se7BB2GhpPfXkhKNX2XqPqFy7FE8GcIAaKjexerkcriJz7fGbx6cZvMahwzxwk5GVdgdcQjf28Xs1FyjdDyKW/a62RgqLEoqXKsOIznPXF67zCQEqJzTKXqsqnGZFBQsrWPBt9YR8UoE2HwIoKLxzHzSt8NwxCJQUMYpsJmVTwQg7msl616Ot5jzVp3nvhg9Z+ZVrie360H5tc1rGWkUxdMK0t5n52DVFgvk2e3hD3GRJ0Js6lgcYa7dxLM8xvB25Gf6RBfef3dtBubq4uT4mm5Wnx5cf/zFsrdmW7al78S7/HgL1mnJaG4/LOdIjQ8PBbZVjVP3TJu2qoMhyHhp6RXNk8/VNIjuKGMXXDew2wRF2RIsl0Z0CCSyyf0bcQ75PD1sUtrSWqNcjYBDguexruG8k9Z+vo371mwXTQGQQQX23+gmyIQ+6qPvPbex3yaHTieXlUIggn4TVfJnp0lldaSYkxDyzcN2CstP33pa13L+3Pi9ihLzlvptEaWWSRnrLwB39PeABF6kwReoIn9mSfduonSal5jePBlbspT7pH7wCdp1cTyM1iILZdkCr3RQj3aPSSEnVW8Exz46AA7z9Rj9Laj/v0lxhF/H0PLsL7ZnwiPapabh8UV4ODg6OQBKbRqkQD5p8MGlfOujUSBWQEzdLwVMbeqLbXRgRqeNRM5x+4/y7ZhTeAQOU06zhHXO8QC7IYla6PuBH0KTqTSr92zMx/aaWS9lTPVpu6nU1y/4Ii8mUgTm81f/oJwuiFMX9EaHQIUfxRBUXezY29s0G7v5aM3euUXgkqodKiZLsK6Lyqaw7729Tjfe1RcNybrZnly7Bba2wVI6xYVBwZ6TTTYxEj3Dio9agvS6bmyOmWwvEg2DupA7rMGW9TPgIGZpnutoUnjze4104LOy5ijgjwnhcNEWcH6f+BbQB8VO5PNGc8lbjrA7nz0vfVlCdw3sEcf6TJAjr8hnRvRf6ySfo8jX05bsl5mV2T1X6JhXORd/RdH2wgE384sHu7kiIj3b4159EwzCf5bXR2QiTPCnP63RcxucfTo+GyAQ5P4c682D7jeSUoFi3r0g8vlX7lA4EBU9cy6VaTvNrYdhf3fUYZBUN7WufPECWZRPAuUCOalzxF8ndgcxZKhYjzLsvaTNn1vTEkQ+MFtzTXN1vrpcuumPt5vrURJtb9NjO1vzMZMJurla9zT8/ivdzG0OPORqzi/nJ4fnn6/pMyxpbRupIY2USLPVRVa1fnvTz9yTWT7gd0Q0vn7npekiBlFbOKZmvq8KLy4HFweXg/DsKPz8l/Dq+nJwcGovHZpzxr4w92/59u0SpaPzz2eHtqenuWwaV1ap7cx/ZYXrP+3qLvXy81mz3KPjyyv+mstaeyeJI/CBuQ5cjWdqHsFtyp01a7oAZux8L9ncUeLjBCMm7RW7tv6rN+2MvByfXUBaeCLtBc0q2Hsh7O972gXOLP3/qqiam9vyMSVpS797aHoEWcQuH1xe2x4g7OrR38tLXV61ep1LT+Hl7q76D7f/hqxbQw3EwpA+dw7DgD41JQcahnZfHOnG/wGsgIVi', 'engine09_main.py': 'eNrdPGtz2ziS3/0rmNqqIxnDtKQksxPpOFWeWN5xbcbO2s7s7qlULIqCLK4pkkNQtpVs7rdfPwA+JErO7V3dh6vygwQajUaj0S+0FK/yrCitTAlLbeDPP1SWCutLnC/iRAorzoS1DNUyiWcAsFyXcSKssLjPw0JhN42GTo97VmG5PFoU2crK4QnaNYT1CTv0c7pe5RsrVFaam6Y8TOfQAD/5nMerKM43nirDUhkcM1mGwnpQwUCFq1xYT6FSslCljNNgHgNoGkk9+CGRYZF6K1kWcVQhKLIoCNdRoKKsAOLDR1mE9zLICxnFKs5S7jg6+nV898v1+a0/sS/k/Ozx/tePn2yBz5+K7Ll6OZuHq+rl79l9jC/To9vxGIf2xNuBePO290fRH7wRg97gh+nR+fWvZ5dXiPfq4uTz1e1fT65+7r8DBPB6l12dXGZ3/PJzdle/fLgdn3y4/HByeX476PV/hCluzu7GgMXr9frCo99306OjuVxYSRbOg1U2XyfSScOVFLgN7vDIUrmM/PZuedgWIL8C3OsgyaKwBC7wQFUWDg12R6vtgTwBD0UkDv5xRyA/uktNEMfUX42wx0OqZOHJZ5hPE7dyR4Us10VqrZhytQwdQ+zS1yLnQevg3Q+Oe2Q9xeXSynKZEpSwi5ntorwsAN5aZIUVLdfpgxWnVlzKwknC1WweDhdeIcO50+8N3r7GP66Y2bY7XHrrfB6W0qFBgF3TsvSW8nke30tVwpxEl36L58odGqg2cWnuKeAO/g9VWBThBqHFvNzk0ofGOC1/eOu6XpnNNqVUDjzuzEL7FhbRMn6UvEBLPgPrSjmvn4IiW6eAuW4A6Q4ThSwrsiflf/2m+aSPr/cfcX4B/5mxyK0vyC3cG+V/8fB/EhMRlhUDEZL3XrmvfHxWstTvsPQwVtK6WadlvJLjosgKxz5f50kMMiMtTTkdemUjOjy4/iSlnUlxVwgRzpJ6EhaBZDr2abGGQwega+Wh4rHdaU0KtgMl/c65f42VitN7C9DDTsf362ytrB1smpC18vGd5FA5X1gmsGPSm7p68XrcPazZ5mcbJrejbJUnspQ2ztSEWYTAWLMDCNrrpPMyzdclbVgBjSkov6y0KqQ4N2EIeE86GQZckqCX5uqUQQObui1QmS1mNjjIVJE8MELAhYMac6EgWGUHX7AXDv2D3PhOObGXWTLP1qU9FSDH2KCknNtTV8AjqNdlNq+7IiApxnNlM1uResBTzayQieVhHutuFnXs3pL+PVx+DJN4zrMMLfuYFkErpHknQMXUL6HhKUwBl//VWQolViJyiT/0gmSayVijYEsRpvfS2T5x37TQAKm65ZXPqDvp+5waBKcrLbgVrzSJSDXqXJR8jZJ42Ny+euNaYhEtMyVTlAsULg2HvMRXfG4KSZp/AQUYZUBeupaIC3Yx3CMHk+HJ2+mxkSwtFAi/LRfcZkSD34x0bEkCaqoX1ImSCe8CGAuZ2AYBLgdRs4RQX2BAA1we0BOU4JzIEmyDXn89oFwWUiHZQQTCMivI2KHj8I8MlsDWrFynsDm2e1DKVLYuInmSpcnGyouszKJME1ltKhxlLXnOa1g3yNn0JZHCcwsujip9tCZy7tTIBPKdLZpVDp0TOGpMaJCjO7PIi9472IhGe7gGXweatg8lKDzNTN4jLToNmFc+0tAahqxsg2fpIr5vwOr3GlDBVpYBGkqparh2ayeP/xqnqSy0BMD+WHBewKmMluZ8ACtYt5ChA/OKQuvEmfczGtfL67YaE2GSZE9BHkfgEfoXwGdJdjCkUwRm//d1XIA+AMYBhQHYbfC56DEJZzKp3sgxxLem6DB0s6Ua1Gysxuq90ZPwix5BuwJmtPC/PgzDyQNLC7kzhkZUOdRaZAlpA6a5PVmFGNhLKGEyH9BOcNCxTXNPR5tmU4sCEAwA8WDrV2B5cT9RH6Hzwy7BxjVtwHmY6Pe1pF7djY9kKGXZANhg99ee6H/DvoZD9crvkhYibHrwBMLgU6a7KRbHNrzgYNesBV/AwGiWMEcs2o8mB/QGMQdIrWODRw7cK/TgFiBj4MPRyqgLPD/s2vBDpWrB11OLOAUH1CE41wPxc7qVybiyNDTAMnMwMTvSjhsfihntu9O18a6omo0Sg5ZO0XCNbCwsdk8hiAJa+nMHmRIaMRH4NjNvrqfiL7JzKbe4fVYG0VQS5qwIUcmz0SXTgKigXVtM7GwY9u+1mE+kGlqGkjC5tf+O74L1KfvVpDocYBz6yCugI/TD0Womwpk/I3pWHcpqNfsuVcXLRpBKSeHiq4N7QKd8pxY5pDCGR5VNxMADw44ANEWYOCHqD1gh/HUPEA68OgX3AZRtU8U+6ICEpAIOm/MsUoovmMFOz8NZn32/ZwEZWmwdjMy9PF84Xm/wTsCQk+fjPmjfvoFOu6Hf/xGgj/sIj9tIM1dmms+QEsAViSQQsX5HtKXhqoBLHyX0V3x0TrgJVo6YXqPcECqYcZTHj1np0+vkBLQuuilMqm7rT6u1M/GADESzDBcYZdJwQYdoActlCGp0xQMvRwIT1xhqbsDHLHEdua9+8stRmRvi1Hrl5P/mbHy/jzQtOjp62FGmzY7/bPQsunr6zZMxj6PSCcsyjB4C8BtBL/llfuqU+fEiBV8epRPcCH8BbYv8uIS2KitSAeYu8oeemEc9T8AqylwAyYtcAH1lKoAWQMmLD9fzuEQlJmbguydSgI5AlBTko89NLqIPACNWEz7DjSIwUZOuQzEdRWq7y+jvUUmjmkcN2lTVVsFFqR+pSYTcmyJXSzUp6eXIinzilPZufSASfFq/opT/mWX4+r/YdqP8fd6VaHlQ7HXP16tcOZ0OlkBRD4A9yr8r1tIViCwJZjKN71OyuFGKjbS8RivuH7mAfivl5ZQbUSrsyovI35P+MkCVa9nvQiJW4TNJjNfrN2DfvQD7DkUyy/3JdFTLF7yoZbwo4QHlFYMaDNH4NBWweeTSIwYbNRo7QahhmWewp+ukhH39xsoXTzn2UpYM1eQsVNKP/kkbW+PyEU405QgQZckaH/3+qWHuF1lkAB5kqQxUDIQF798F6xzieb9/Agt6/dqpgMlUg+T4ZhETnEOnNPDR9zFnR5G7HvLv/V6vZ0I6GDoxgTHYITtO1XqxiKNYpmZ7m7GL/YLlrHhpmWAb58lAcQL56dxBvjBXdJLkxQlB1TQix1Kw/nvwa50dpayvR6RNjMZcyRBXC2oPMMhHv1KKLB+lO0oysYz9yurIx4m9yOEAmIdjfChTjiaBVYD+J5zouC9P+oNuM0dhWm1PrMc4S/Q6AMkfrOsivo/TMKnSV6pEBQEj4BFDaFirQnpK7Ky0vpVnEKduPOuGPXNLPodRScxOwTFZU9ziHXXuqFn4nHMdr3wCqEjkECpMN8iAhyl3N4MBgM3BF1jgnxJdugWyhAbNFGju4oRnpKW34lw+PT/tZ9eNEZaKmtMoA4jOCIxcbXyhzSaxPyBW9apFhdwvRTeVYEmKGiygqSXrP0EEKW7jlvo8B/I5kkr5JCjantnTEzrlDSiaaAvodBsIY8s4De6zMNkC/XdWGwuMKuN4HkQxqAN48UGA243L+H7pL2Px+rV8dFngzuPwPs1UGUcWJg+GlopX6yQ03v+bAXv/p7XIokSCjhaYZ1hlmCzKqkhhJkEsJCUTwyJWLHOApD52GquDp05BR6nAZ0IvqdHpbjeB4zSKADhKvw/4CMPooBSBCJp64M1AKwIJuJrnHQgRiAo7IA7b7kJciLMyD7tiVQuHpgFkyai6oBaw7aMltsa9GXBKEeIFcGKQqqofDlMg8Vj4eKyg7wQcOvYVWeWVYQxmeD0HgfEfjCCzf8FeeMDQDa8MnA4eDI6iFrXDA8BzqQZwJB3oibRm5uF+OxpHM2TAW9NswYGJdw3hKPKGJaQDm2zcUuJvBj8hs+pFHBwMe9w8PYJdzxfgW24q7tvuRNRHWkPSoEIemqYLenuSaIlZuTl6QyhkIHhql7zG7gAD0Zy98h1VscTtxNKafT8GEvx6T0jCUKXJOd5r6r3c2o2STSqqlwflm5tRNMLgAR5Z5FWZ86MdIC07D4puSmJURhrpg/KqJldouMYdqwbrunXVM4JnyVztxG6IY8+7D/vKbncfcwv1xBrF907cgc0Fnft1YX/F9M634Pevv3+zhxXf4KyB1Uuk8yh+P0Xny3WrrJp41PmV1jmzBfqBjnFDTSOslw3z7zToXa8n3uPvO/h9j7/v4Pf99FsjCMvyOqoStDkmSxGnDoeG5mrdOyvu13hh9AnfCgdCVS+c4x0hNzv2yclTVjzYosphUoTQAaZW2YO0RUji6Nsk1AH4KdJ2R6GfezQfjlAwy5OPN/VO6CFyePVWD/MYb4ILwKatsHwG7gfZA88I/rzyZPoYF1k6sS8+Xp7fBuO/feoNgpvr6zt76qOb8MTZn9AjYob7Rtz+ev3nMQyx+/YRhZgNSC/PcqcDWlyBaw7oV37z8tsGJ7g3sMXTqS1T8PJkb+DlG1gxHGNu2AX/Mci0R9gcV7cyglQ+7UPwPsCNTmVzuGmjwRD1wvzK14UAk2F/2mALB9W6b4SBp/KpmqATjnpGfCnmD3a63/SO0Puk2JFvu2b6rgvFlcigV4WvNBe9UoZR1z98w7U2EDQrIF7ABaHYH6zbElwXqzdEW44CZIUWJvIS8llWWWpBTAxGvbSiEFw8iooovUvxG5Z9wE6Vcbkxnk4JbivmwMDPgYAM75p94PMCRoANxe0OFtAri7wALau8FDh+hKHPPW6iQYZDttv4fquG1RdrCKrlrbq/hd4bHW+e2hrOxtwUi8jhId6XOLc57eh9OPvwy9ijowTn7pWvV1S17Ml/m7tS5tlpY8FWHsbFCFQNHXBrlmG9BOg1ucoe9avmI28HZYKPqnu1LYKG1a14m1OmX1+HN9l1mHRkAYRUWltxkGW41hCA5g7a7AeiybRv787+NA56waeb8aezm8urPwVXF8HnvwREtQ2u91ottQZceVresK6BlMgWjpvx51vEsDv2qH2rpJmy7xIJxRf59Ozj08R+pqsVft7A81w/z/FqYZH7Bq3e6i60tB/P+v5lgEx+5juGSX/KVywr72J8dvf5ZnxbXcU871zOLPKdpnmrZfu64vnQVYW5deFdqq480HGXtEnVVSaIfBAlYKf9waC3o5L6AzC3R6a8YwME/fh6d+hL90ftvjn2YZNWm64JoJuO1tz3ly7lRCkt6LrNsoJ64kqjaVzUQCPIwgMBexLqrZQLZgyyGOiMv8jTeYa2HfQd59pGdYFBJftYCvYXaxYm6M80ECBj8bSv0W1sZLsnpvxoloQPcjDjTDilZagKhqGK7KlRcST40i1Amvz+oFWApL2fJ1zj87TWCFu3CoYUAbLVyYQP2wpJta5FjAbp/eh/tUmSAq6fsodY+qWPmivsJoYWiD42AGIseqt729KL2oB3g20ZdkElDcoe0j9BxQzwRv+ELmaAd20bhalQGeqiLJvdrKGW+G9HB9WnVq+NqosdULzFDkr5DFsE4lqzb+9toi4w6v2IMoS6trKfZh9q60CveKkWFmW8ANeQjt1aoY2OEggtLPT/LHD9QFqzglwXUqVts7Ct+RFJxdSD5mCMnUhAm/KanASCAMx6PMoUD8boBeJ2Wf1UgGJjBjbS7DUfRZzisz9AB71lIfroX45vLn8dX90FgBJab+5sQRdH7FG5r51dj+utK+xmVQ87K2rLwGBScIdDlB06S60wwV3fVAVi8yo1GSvrQealhQf6BE7sCT7Q0UVD2mTi++3tp7RQsRmiOyjgVxeHtUoODUlCL1Boqe5aJaDDuBSIcUyd4c/hXJcaitY+C+vPcqOffkNzoZ/H1xf0RGYUkJkEuNPlPr0PtN8JbhR516c2r3DLf3MPS9xOtjdGd5UtG+d12VFHScL6m/qwIIc5x6uHoveLZcZAORKuhedm/PPny4/nweXVh+tfP30c340DkieQoLObD79c/jbGYC0vHBjmtjyWSiDWaRKnD06thDtFpe15D015QO198/W+JquWZSDk4vrjuS3QmW8JpUXXFqwXPayiXGAObyWeCNSoPC0TWhwo6Wvc3np0eH9fyHtM6MFg0qSsQQ2SenTTNUOXbBwYvp0HLaIr7rVo/l+T5oNn35BUHX9drWVe3TZRf0AxZwWgiWnk0xV5uJzDNr44uCrkE6QKRZIkUEc4mPKvLk5xawGDBypthVKwc7Fqg88LkdD9sjQ1CE59wSpA3kqNgumvELE+b9RwvWrs5Yezq/PLc7xGmzCi6SSavuQBNZx6owsZ97qoLl80y69vLv90eXX20exw8NvZR5rv3N7DX30snO1whTjrvBQN1hrCHbaiBPTT/e/w+ve596NmCNDw9F8iaG8I8H/u1tssvKBzFxuQ2FAH5e2AzKIgPirBH0w21hyAUb5JFaKO1COwCMf7/+W7MlajOcz1LUYw73/A+9uDBV7VgXj/gwDgE6SXnf7K1x9ZMH+EN3xA52m1IbuKvkPPl4V4DAXWHHhctAR+TRwB5zYCF4wa3B2hYcO77qaiRhUSgAgENMwBJBuEnwM84uXaJ6xHbFTssVoS8Ggq0dDNsUVZYIZUZ2+o2OwxrCvSSknPXH0mkJad2jR4cadc4hgXqqTLG6dhfLBSf2osXTNJxbaOtWA9auZOR1z7Rei0lqzqAnltr3S1CeD+roKvT6ww5hbBgv+5WMhCsfy33NiRRUns5XaCA0hnP+LD7W98GV+nnhss/W/xbFhVLXYIvl5ff9qq/xSmTPN7RphCGrGZwLA9AfAWZ+ryzNYx0lr/7OPHoNL8t58+Xt7dBr+Byb24NHqfxN19zZ9BATnfMbI6mUQV2LBy0OSPMX4ApM4jZTveHvuKsQQtBh6K9OqQFAJSA94KFU0j+fX/QsD6ojkS9n0Rz+1hlVJuWtxmtnW6HV5C+EnOasDj9kS4rdzzt3rBVe70X/a1Ry1UHl8P6LuC7esB9Ou7wOv8qdu24/QphtaA7wslNTpLfxRgOw7SmrwbM03cCMfbUN2x+KE8Zx25NuonMIRtBLDoZMuiGTOzAVDNfAkTcDiY3R/KDlruLOzu+PzTzfXfgpvx3eercTOy1al+9/Xb74liv8syaUq25twbgtSnYCcC2ToMO4EIDIVQ1K8x7ItCthC1MAj47YomGLnQDDKTvz3gZleIDrnZnYf++91sHazuuNY6btBkDNkIUhmR3ExxEfSwLSdvgrtfbsa3v8DeBGefzy/vjGi0Q651uUdrmDuREYDsUQNZrkzZX1Xxp6jkrwp1tqMc4Nqjb1PBhpzDiXq29eeIJoOp7ze3kqM5VMVhYsOwsC9mfRH1/UagJB4h0s7VsR/2iYpjf9bXZBz7UX+LDuBUk45OhHq+F9DOF34+984hPrgowhVEpzk4ZPMFeLhBpB4d4NipnQEvw5I+0ZORHQEP24Nem472swk/WnhwMreFpS5gqWs6id6XUDG5bWRUNRJwCcHO8COLlu7PFxNYiSmH9TVDIAaipEUTHbUE2+vswqw/tO1v12iIusKDwl1eIDzzh53wf17g/7pytfmCn7PeqRZrt9EpwssaTtv7vIp7OPH5DIIX84k2UX34TbRqU0WoguZSJnolUy7WgE0ntC22cFMANmMVFpsuduguXw/uoKZVH1tPCjqQoELkkSrxk3ijap4sWa9S5HFge/j5N+fBrasMt6Cm1TAMWEpeo9OWFq5henk1VSkgM3fCLNaFgs1Kwemoen5xzbTUCjygTwYq3+HPI8LaIXIEZ4uJw/Z6+1K+RoJeHmQq/1AaqPrPsbvLAhHtemW77gGeGCSBuf7pYgje2upjpGvMW4oNFCYC4J8J/ulklUdlYg4jnjSWpxnQElKz9bY+tRiq4UXe4fNmzti+swWyReQdFqwmiFb8LK5c2/o/mX9ILGJsELbAZpahPfXbrVo9nbRbmwZmyjtyaFcZMkAwEDbO4wWUQe7c36aenZT//GrXHwa3h3az6OQb5/PbiTreHMedHncMrgpOGkNr56Ma2l7AcgMowAUHgoBm/fFTkNHVDJyReacKYpn7aiqZh/VHtkHGskX5FGJJEe4WhmSNKKWKq/h42cMqrVd3GR/TfJZ+uJVptcusxENknNKgdkq3QI+1I1uNZO5Uu9uiAuAAAv52E9BEs9hZwrZl2Dly1Yefq+Hbpq+BCO8qK+11CLqCghGPmKuLYQBnLhZ8edkdPttYDVlPUWekA2Bq0KyqtodU4mXX8Ys9NGEGRJAsQTvfs7AnPmGYVnTCCC6v7sY3nyAgOLu7vL7yyueyjcO+a1aH1OXQXHmPNzc5587AEXa6q/RH1ircWOCqg5ZbZpjoyFaE7eLTjetZF+bjllxgjSgx7DS11akElxrDsYWEUDWSesCbgVWVDOOQEKKdVMUwM16z6mpuLO4jZBiPwi5trGzRyg+98axrWIzW8ogH5DTEw8cZDTzKrF2seVUxrkbsclllOEskhpKYeQK+APi6OKlDNkwpeNYY7TF/tQDfOFgwf6NW3CxYMhyeMGsZIzfiCBso72TJxYK+HKjAeBZWgT85xCNhgpkA6wJmtti4aJcJv0WIaslSUAagTJBtTyeJfJSJBeoFtoC4yp/ywO/74QgddglrVJE7OogX1tMyjpa0iUgnbr9n/bxTJWFxxS0JBKCUdFVcOYenZ59oNKBPsg1xH+gC+kFXedYlKrlWCRVlYTnNTd/TEeblumjSF+3JLuFtNezHOkmshQxpEKYSPeuOuaOzYbBrjT3WBxJr3K2rzDCylvZFXKIG8Dg3f5lGyXoO88McoJwa1ysVy+hLj+jLJdZwNKCB1sMKRNTfqqA/L0krofAA85GYstc1xk+nTleMd3t9cffXs5txAIHqXXB99fHvVM+2c6d2ID6kAe6eL6nh2YX9ZIuq6/JTcD6++IhXQgKFF+hWJE3+u/r7bDiHyl99MQH9Ir4rszWtLk4X5PPyly4gHq+4T7KZY78G9676pPTCixV9TxLEg19YVzkLgd8vlJDmCcrMeaLPeZje6hayYbR7P57ax9WlrvmWkuZ3fEz2VCeKVi6R8pH0XR7icI6RV2lIejql73ay8Ro6m6+jeBbjST6tvzBlmx+dN/KDKt53Pc0q8xUhrekWuzPhNVnAN28k+TD1Yg8fmtU8ortoR3RlOhuNFDHt5UIEBr1e+u51+fvG7e+OVXPpG7JYZtv5aRCWgL5jJwjATQ+IiCCwh1zlffRf8ooFcQ=='}
for filename, encoded in PAYLOADS.items():
    data=zlib.decompress(base64.b64decode(encoded))
    target=WORK/filename
    if target.exists() and target.read_bytes()!=data and (WORK/'stage08_identity.json').exists():
        raise RuntimeError(f'Engine mismatch during resume: {filename}')
    target.write_bytes(data)
    print('ENGINE_READY',filename,len(data))

In [ ]:
# Run stages 0–3 in sequence; stream logs and retain completed candidate checkpoints.
cmd=[sys.executable,'-X','faulthandler',str(WORK/'engine09_main.py'),'--work',str(WORK)]
if SMOKE_TEST:cmd+=['--smoke']
env=os.environ.copy();env['PYTHONUNBUFFERED']='1'
logpath=WORK/'experiment08_09.log'
start=time.time()
with logpath.open('a',encoding='utf-8') as log:
    proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in proc.stdout:
        print(line,end='',flush=True);log.write(line);log.flush()
    rc=proc.wait()
print('Elapsed seconds:',round(time.time()-start,1))
if rc:raise RuntimeError(f'Run failed (exit {rc}); see {logpath}. Completed candidates remain in the work directory for resume.')

In [ ]:
import pandas as pd
out=WORK/'FLIDS_Experiment09_Results'
result=WORK/('FLIDS_Experiment09_SOFTWARE_TEST_ONLY.zip' if SMOKE_TEST else 'FLIDS_Experiment09_Results.zip')
with zipfile.ZipFile(result) as z:
    assert z.testzip() is None
    status=json.loads(z.read('FLIDS_Experiment09_Results/run_status.json'))
    assert 'experiment08/FLIDS_Experiment08_Results.zip' in z.namelist()
print(json.dumps(status,indent=2))
if not SMOKE_TEST:
    assert status['original_training_trials']==320 and status['new_training_trials']==80 and status['total_candidate_trainings']==400
    assert status['final_models']==80 and status['final_operating_points']==240
    assert status['all_evaluated_thresholds_and_counts_match']
summary=pd.read_csv(out/'equal_domain_summary.csv')
display(summary[summary.target_fpr==.01])
display(pd.read_csv(out/'fpr_goal_coverage.csv'))
final_download=BASE/result.name
if final_download.resolve()!=result.resolve():shutil.copy2(result,final_download)
print('FINAL ZIP:',final_download,'| MB:',round(final_download.stat().st_size/1048576,2))
display(FileLink(str(final_download)))

## Output and resume

The final `FLIDS_Experiment09_Results.zip` contains the complete original Experiment 08 archive as a nested ZIP, new FedProx trials/checkpoints/scores, threshold/precision audits, paired before/after results, combined trial table, original cache/fingerprints and code. The full work directory retains intermediate checkpoints. If a session ends, preserve that directory and attach it as Kaggle input to resume; Kaggle does not guarantee that unsaved outputs survive a terminated session. Keep the same cache, engines and settings.

The 320 reconstructed runs follow the original protocol, not necessarily bit-identical historical results; input row order, library versions and GPU kernels may differ. Seeds are nested within four domains. Float64 score storage does not imply float64 inference. Flow-level binomial intervals assume independent records. Balanced reservoir changes deployment prevalence. The new FedProx grid is a limited retune and adds historical search effort; no global optimality or target FPR guarantee is claimed. Experiment 03 historical score files are not reconstructed. Smoke runs use synthetic data, fewer rounds and fewer folds/candidates and are software tests only.